# Task 2 — Incremental CPG parser

```{admonition} Evidence summary
:class: note

| Field | Value |
|---|---|
| Evidence source | Hash-validated Stage 2 manifest and Kafka samples |
| Command | `docker compose run --rm ... parser python -m parser_service.main --repo data/datasets --mode full` |
| Processing unit | One source file, flushed before the next file |
| Identity | Stable file, node, and edge IDs |
```

## Approach and reasoning

Command: `docker compose run --rm ... parser python -m parser_service.main --repo data/datasets --mode full`.

The parser holds one source string and one Python AST at a time. Generator-based
extractors emit AST nodes and CFG/DFG/CALL edges directly to Kafka; the producer
flushes per file and the AST is released in `finally`. Structural paths make
identical source content produce identical identifiers.

## Key implementation excerpts

```{literalinclude} ../parser_service/parser.py
:language: python
:pyobject: process_file
:caption: Per-file parse, emit, flush, and cleanup lifecycle
```

```{literalinclude} ../parser_service/ids.py
:language: python
:pyobject: make_node_id
:caption: Stable structural node identity
```

## Executed evidence


In [1]:
from pathlib import Path
import json

ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "screenshots/stage2_manifest.json").exists()
)

def first_json_line(relative_path):
    path = ROOT / relative_path
    return json.loads(next(line for line in path.read_text().splitlines() if line.strip()))

manifest = json.loads((ROOT / "screenshots/stage2_manifest.json").read_text())
metrics = manifest["metrics"]["kafka"]
metadata = first_json_line("screenshots/kafka/sample_cpg_metadata.json")
node = first_json_line("screenshots/kafka/sample_cpg_nodes.json")
edge = first_json_line("screenshots/kafka/sample_cpg_edges.json")

metadata_view = {
    key: metadata[key]
    for key in (
        "file_path", "status", "num_ast_nodes", "num_cfg_edges",
        "num_dfg_edges", "num_call_edges", "content_hash"
    )
}
print("metadata sample:", json.dumps(metadata_view, sort_keys=True))
print("stable identity proof:", {"node_id": node["id"], "edge_id": edge["id"]})
print("AST nodes:", metrics["ast_nodes"])
print("CFG/DFG/CALL edges:", metrics["cfg_edges"], metrics["dfg_edges"], metrics["call_edges"])
print("metadata/error events:", metrics["metadata_events"], metrics["error_events"])

assert metadata["status"] == "success"
assert len(node["id"]) == len(edge["id"]) == 16
assert metrics["metadata_events"] == manifest["run"]["files_processed"] == 138
assert metrics["ast_nodes"] == 133263
assert metrics["total_edges"] == (
    metrics["cfg_edges"] + metrics["dfg_edges"] + metrics["call_edges"]
)


metadata sample: {"content_hash": "74ab176247c9ba61e09843280c18cf3a9ac690403a5d35411a75d097db721939", "file_path": "src/datasets/__init__.py", "num_ast_nodes": 57, "num_call_edges": 0, "num_cfg_edges": 15, "num_dfg_edges": 0, "status": "success"}
stable identity proof: {'node_id': '57d761e9567d5b95', 'edge_id': '07474daf0400bc31'}
AST nodes: 133263
CFG/DFG/CALL edges: 18097 8586 11458
metadata/error events: 138 1


## What this chapter proves

| Requirement | Evidence |
|---|---|
| One-file processing | The embedded implementation shows per-file parse and flush. |
| Complete CPG categories | Executed output reports AST, CFG, DFG, and CALL totals. |
| Stable identifiers | Real node and edge IDs are printed from captured Kafka events. |
| Error visibility | The canonical run records 138 metadata events and one controlled parser error. |

## Reflection

```{admonition} Closing reflection
:class: tip

| Point | Result |
|---|---|
| Worked | Generator extractors and per-file cleanup bounded parser memory. |
| Failed | A previous sample run covered only part of the repository. |
| Resolution | The canonical run uses full mode and rejects evidence missing any selected file. |
```
